# DeepFake Sentinel — Phase 1 Training
## Real vs AI-Generated Face Classifier (EfficientNet-B4 + ViT-Base/16)

This notebook fine-tunes the **EfficientNet-B4** and **Vision Transformer (ViT-Base/16)** models from your `deepfake-detection-platform/backend/models/` package on the **140k Real and Fake Faces** dataset (real photos from Flickr-Faces-HQ vs StyleGAN-generated fakes).

**Output:** `efficientnet_b4.pt` and `vit_base.pt` — drop these into `backend/weights/` in your project and `DetectionEngine` will automatically switch from simulation mode to real inference.

**Runtime:** Set `Runtime > Change runtime type > GPU` (T4 is fine) before running.

**Estimated time:** ~30-60 min for a few epochs on a subset (full dataset takes longer; subset sizes are configurable below).

## 1. Setup — install dependencies

In [ ]:
!pip install -q timm kagglehub torchmetrics

## 2. Download dataset

Uses the **140k Real and Fake Faces** dataset (Kaggle: `xhlulu/140k-real-and-fake-faces`).

Structure after download:
```
real_vs_fake/real-vs-fake/
    train/real/*.jpg, train/fake/*.jpg
    valid/real/*.jpg, valid/fake/*.jpg
    test/real/*.jpg,  test/fake/*.jpg
```

In [ ]:
import kagglehub

# Downloads to a local cache dir, returns the path
dataset_path = kagglehub.dataset_download("xhlulu/140k-real-and-fake-faces")
print("Dataset downloaded to:", dataset_path)

import os
for root, dirs, files in os.walk(dataset_path):
    depth = root[len(dataset_path):].count(os.sep)
    if depth <= 3:
        print(" " * depth + os.path.basename(root) + f"/  ({len(files)} files)")

## 3. Get the model definitions from your project

Upload your `deepfake-detection-platform` project as a zip, OR clone from GitHub if you've already pushed it. This cell expects a zip uploaded to Colab containing the `backend/models/` package.

In [ ]:
# OPTION A: Upload your project zip (recommended)
from google.colab import files
import zipfile, os

print("Upload your 'deepfake-detection-platform.zip' (or just a zip of the backend/models folder)")
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as zf:
    zf.extractall('/content/project')

# Locate the backend directory (handles nested zip structures)
backend_dir = None
for root, dirs, _ in os.walk('/content/project'):
    if 'models' in dirs and os.path.basename(root) == 'backend':
        backend_dir = root
        break

assert backend_dir is not None, "Could not find backend/models — check your zip structure"
print("Found backend at:", backend_dir)

import sys
sys.path.insert(0, os.path.dirname(backend_dir))  # so 'backend.models' is importable

In [ ]:
# OPTION B (alternative): if OPTION A's import fails due to package __init__ issues,
# import the model classes directly from the files instead of as a package.
import importlib.util

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

try:
    from backend.models import create_efficientnet, create_vit
    print("Imported via package (Option A) successfully.")
except Exception as e:
    print("Package import failed (", e, ") — falling back to direct file import.")
    eff_mod = load_module("efficientnet", os.path.join(backend_dir, "models", "efficientnet.py"))
    vit_mod = load_module("vit", os.path.join(backend_dir, "models", "vit.py"))
    create_efficientnet = eff_mod.create_efficientnet
    create_vit = vit_mod.create_vit
    print("Imported via direct file load successfully.")

## 4. Dataset & DataLoaders

`SUBSET_FRACTION` controls how much of the 140k dataset to use. Start with `0.1` (≈14k images) for a fast first run; increase later for better accuracy.

In [ ]:
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import random

# --- Config -------------------------------------------------------------
IMG_SIZE = 224
BATCH_SIZE = 32
SUBSET_FRACTION = 0.10   # fraction of train/valid sets to use (0.0-1.0)
NUM_WORKERS = 2
SEED = 42
# --------------------------------------------------------------------------

random.seed(SEED)
torch.manual_seed(SEED)

DATA_ROOT = os.path.join(dataset_path, "real_vs_fake", "real-vs-fake")
assert os.path.isdir(DATA_ROOT), f"Unexpected dataset layout, check: {dataset_path}"

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# ImageFolder expects class subfolders: real/ -> class 0, fake/ -> class 1 (alphabetical)
train_full = datasets.ImageFolder(os.path.join(DATA_ROOT, "train"), transform=train_tf)
valid_full = datasets.ImageFolder(os.path.join(DATA_ROOT, "valid"), transform=eval_tf)

print("Class mapping:", train_full.class_to_idx)  # expect {'fake': 0, 'real': 1} or {'real':0,'fake':1}
FAKE_CLASS_IDX = train_full.class_to_idx.get("fake")
print("Fake class index:", FAKE_CLASS_IDX)

def make_subset(ds, fraction, seed):
    n = int(len(ds) * fraction)
    rng = random.Random(seed)
    indices = rng.sample(range(len(ds)), n)
    return Subset(ds, indices)

train_ds = make_subset(train_full, SUBSET_FRACTION, SEED)
valid_ds = make_subset(valid_full, min(SUBSET_FRACTION * 2, 1.0), SEED + 1)

print(f"Train subset: {len(train_ds)} images | Valid subset: {len(valid_ds)} images")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

## 5. Label alignment helper

Your platform's models output `logits[:, 1] = P(fake)`. `ImageFolder` assigns class indices alphabetically (`fake`=0, `real`=1 in this dataset). We remap labels so that **index 1 always means "fake"**, matching `DetectionEngine`'s `softmax(...)[0, 1]` convention.

In [ ]:
def remap_labels(labels, fake_idx):
    """Convert ImageFolder labels so that 1 == fake, 0 == real."""
    if fake_idx == 1:
        return labels
    return 1 - labels  # flip 0<->1

## 6. Training loop (generic — used for both models)

In [ ]:
import time
import torch.nn as nn
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


def train_model(model, model_name, epochs=3, lr=1e-4):
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_val_acc = 0.0
    history = []

    for epoch in range(epochs):
        # --- Train ---
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        start = time.time()

        for images, labels in train_loader:
            images = images.to(device)
            labels = remap_labels(labels, FAKE_CLASS_IDX).to(device)

            optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / total
        train_acc = correct / total

        # --- Validate ---
        model.eval()
        val_correct, val_total, val_loss = 0, 0, 0.0
        with torch.no_grad():
            for images, labels in valid_loader:
                images = images.to(device)
                labels = remap_labels(labels, FAKE_CLASS_IDX).to(device)
                logits = model(images)
                loss = criterion(logits, labels)
                val_loss += loss.item() * images.size(0)
                preds = logits.argmax(dim=1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

        val_loss /= val_total
        val_acc = val_correct / val_total
        scheduler.step()

        elapsed = time.time() - start
        print(f"[{model_name}] Epoch {epoch+1}/{epochs} "
              f"| train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
              f"| val_loss={val_loss:.4f} val_acc={val_acc:.4f} "
              f"| {elapsed:.1f}s")
        history.append({"epoch": epoch+1, "train_loss": train_loss, "train_acc": train_acc,
                         "val_loss": val_loss, "val_acc": val_acc})

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), f"{model_name}_best.pt")
            print(f"  -> saved new best checkpoint (val_acc={val_acc:.4f})")

    return history

## 7. Train EfficientNet-B4

Starts from ImageNet-pretrained weights, fine-tunes the whole network end-to-end (or just the head if you reduce trainable params for speed).

In [ ]:
eff_model = create_efficientnet(pretrained=True, num_classes=2)
eff_history = train_model(eff_model, "efficientnet_b4", epochs=3, lr=1e-4)

## 8. Train ViT-Base/16

ViT is heavier — if you hit memory/time limits, reduce `BATCH_SIZE` to 16 or `SUBSET_FRACTION` further, or skip this and ship EfficientNet-only first (the ensemble degrades gracefully).

In [ ]:
vit_model = create_vit(pretrained=True, num_classes=2)
vit_history = train_model(vit_model, "vit_base", epochs=3, lr=5e-5)

## 9. Quick sanity test on a few images

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

eff_model.load_state_dict(torch.load("efficientnet_b4_best.pt", map_location=device))
eff_model.eval()

# Grab a handful of test images
test_dir = os.path.join(DATA_ROOT, "test")
samples = []
for cls in ["real", "fake"]:
    cls_dir = os.path.join(test_dir, cls)
    files_in_cls = os.listdir(cls_dir)[:3]
    for f in files_in_cls:
        samples.append((os.path.join(cls_dir, f), cls))

fig, axes = plt.subplots(1, len(samples), figsize=(4 * len(samples), 4))
for ax, (path, true_label) in zip(axes, samples):
    img = Image.open(path).convert("RGB")
    tensor = eval_tf(img).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = F.softmax(eff_model(tensor), dim=-1)[0]
    fake_prob = probs[1].item()
    pred = "FAKE" if fake_prob >= 0.5 else "REAL"
    ax.imshow(img)
    ax.set_title(f"true={true_label}\npred={pred} ({fake_prob*100:.1f}%)")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 10. Download the trained weights

Download `efficientnet_b4_best.pt` and `vit_base_best.pt`, rename them to `efficientnet_b4.pt` and `vit_base.pt`, and place them in `backend/weights/` in your project.

In [ ]:
from google.colab import files

if os.path.exists("efficientnet_b4_best.pt"):
    files.download("efficientnet_b4_best.pt")
if os.path.exists("vit_base_best.pt"):
    files.download("vit_base_best.pt")

print("\nAfter downloading:")
print("  1. Rename 'efficientnet_b4_best.pt' -> 'efficientnet_b4.pt'")
print("  2. Rename 'vit_base_best.pt' -> 'vit_base.pt'")
print("  3. Place both in backend/weights/")
print("  4. (Optional) Train cnn_lstm.pt separately on video data — see Phase 2 notebook")
print("  5. Restart your server (python run.py) — DetectionEngine will auto-detect and switch to model mode")